In [ ]:
# Step 1: Install necessary packages
!pip install streamlit ngrok pyngrok torch torchaudio audiocraft

# Step 2: Import required libraries
import torch
import torchaudio
import os
import base64
import tempfile  # For handling temporary file paths
import streamlit as st
from audiocraft.models import MusicGen
from pyngrok import ngrok

In [5]:
%%writefile app.py
import streamlit as st
import torch
import torchaudio
import tempfile
import os
import base64
from audiocraft.models import MusicGen

# Step 1: Cache the model to avoid loading it multiple times
@st.cache_resource
def load_model():
    try:
        model = MusicGen.get_pretrained('facebook/musicgen-small')
        return model
    except Exception as e:
        st.error(f"Failed to load model: {e}")
        return None

def generate_music_tensors(description, duration: int):
    model = load_model()
    if model is None:
        return None

    model.set_generation_params(
        use_sampling=True,
        top_k=250,
        duration=duration
    )

    output = model.generate(
        descriptions=[description],
        progress=True,
        return_tokens=True
    )

    return output[0]

def save_audio(samples: torch.Tensor):
    """Saves audio samples to a temporary file and returns the file path."""
    sample_rate = 32000
    temp_dir = tempfile.mkdtemp()
    audio_path = os.path.join(temp_dir, "audio_output.wav")

    samples = samples.detach().cpu()
    if samples.dim() == 2:
        samples = samples[None, ...]

    torchaudio.save(audio_path, samples[0], sample_rate)
    return audio_path

def get_binary_file_downloader_html(bin_file, file_label='File'):
    with open(bin_file, 'rb') as f:
        data = f.read()
    bin_str = base64.b64encode(data).decode()
    href = f'<a href="data:application/octet-stream;base64,{bin_str}" download="{os.path.basename(bin_file)}">Download {file_label}</a>'
    return href

# Step 2: Define the Streamlit app
st.set_page_config(
    page_icon="🎵",
    page_title="Music Gen"
)

def main():
    st.title("Text to Music Generator🎵")

    with st.expander("See explanation"):
        st.write("Music Generator app built using Meta's Audiocraft library. We are using the Music Gen Small model.")

    text_area = st.text_area("Enter your description...")
    time_slider = st.slider("Select time duration (In Seconds)", 0, 20, 10)

    if text_area and time_slider:
        st.json({
            'Your Description': text_area,
            'Selected Time Duration (in Seconds)': time_slider
        })

        st.subheader("Generated Music")

        # Generate music tensors
        music_tensors = generate_music_tensors(text_area, time_slider)
        if music_tensors is None:
            st.error("Music generation failed. Please check the logs.")
            return

        # Save and display audio
        save_music_file = save_audio(music_tensors)
        audio_file = open(save_music_file, 'rb')
        audio_bytes = audio_file.read()

        # Streamlit audio player
        st.audio(audio_bytes, format='audio/wav')

        # Provide download link for the audio
        st.markdown(get_binary_file_downloader_html(save_music_file, 'Audio'), unsafe_allow_html=True)

if __name__ == "__main__":
    main()


Writing app.py


In [8]:
pip install --upgrade pyngrok


In [9]:
from pyngrok import ngrok

# Step 1: Set your ngrok authentication token
ngrok.set_auth_token("2lvid5Eq5LOntl3hZoO6EFHR46W_7YF6QdG8wHr8Tahth59cS")  # Replace with your actual token

# Step 2: Connect to your Streamlit app on the specified port (8501)
public_url = ngrok.connect(8501)  # Default protocol is HTTP
print(f"Streamlit app is live at: {public_url}")

Streamlit app is live at: NgrokTunnel: "https://df0f-35-204-44-101.ngrok-free.app" -> "http://localhost:8501"


In [ ]:
!streamlit run app.py --server.port 8501 --server.headless true





  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.204.44.101:8501

2024-10-11 10:25:32.499456: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-11 10:25:32.542621: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-11 10:25:32.555070: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-11 10:25:34.320578: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
I0000 00:00:1728642336.051279   10703 cuda_executor.cc:1015] succ